In [1]:
import numpy as np
import pandas as pd

In [2]:
df = pd.read_csv("/Users/theerdhasjoy/BIA/ML/Dataset/train.csv")
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [3]:
# EDA

def eda(df):
    shape_df = df.shape
    info_df = df.info()
    desc_df = df.describe()
    missing_df = df.isnull().sum()
    unique_df = df.nunique()
    duplicated_df = df.duplicated().sum()

    return shape_df, info_df, desc_df, missing_df, unique_df, duplicated_df

eda_find = eda(df)

print(eda_find)

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    str    
 4   Sex          891 non-null    str    
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    str    
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    str    
 11  Embarked     889 non-null    str    
dtypes: float64(2), int64(5), str(5)
memory usage: 118.9 KB
((891, 12), None,        PassengerId    Survived      Pclass         Age       SibSp  \
count   891.000000  891.000000  891.000000  714.000000  891.000000   
mean    446.000000    0.383838    2.308642   29.699118    0.523008   
std     257.353842    0.486592

In [4]:
# Feature Engineering

df = df.drop(columns= "Name")
df = df.drop(columns= "PassengerId")

num_cols = df.select_dtypes(include= np.number).columns.tolist()
cat_cols = df.select_dtypes(include= "object").columns.tolist()

print(f"Numerical columns are : {num_cols}")
print(f"Categorical columns are : {cat_cols}")

Numerical columns are : ['Survived', 'Pclass', 'Age', 'SibSp', 'Parch', 'Fare']
Categorical columns are : ['Sex', 'Ticket', 'Cabin', 'Embarked']


/var/folders/y5/lf_stmcj5nv2hnzy3q1b5pb80000gn/T/ipykernel_58836/3803631601.py:7: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = df.select_dtypes(include= "object").columns.tolist()


In [5]:
# Missing values imputation

for col in num_cols:
    if df[col].isnull().sum() > 0:
        df[col] = df[col].fillna(df[col].mean())

for col in cat_cols:
    if df[col].isnull().sum() > 0:
        df[col] = df[col].fillna(df[col].mode()[0])

In [6]:
# Categorical encoding

df = df.drop(columns= "Ticket")

df["Cabin"] = df["Cabin"].str[0]

df = pd.get_dummies(
    df,
    columns= ["Sex", "Embarked", "Cabin"],
    dtype= int
)

df.head()



,Survived,Pclass,Age,SibSp,Parch,Fare,Sex_female,Sex_male,Embarked_C,Embarked_Q,Embarked_S,Cabin_A,Cabin_B,Cabin_C,Cabin_D,Cabin_E,Cabin_F,Cabin_G,Cabin_T
0,0,3,22.0,1,0,7.2500,0,1,0,0,1,0,1,0,0,0,0,0,0
1,1,1,38.0,1,0,71.2833,1,0,1,0,0,0,0,1,0,0,0,0,0
2,1,3,26.0,0,0,7.9250,1,0,0,0,1,0,1,0,0,0,0,0,0
3,1,1,35.0,1,0,53.1000,1,0,0,0,1,0,0,1,0,0,0,0,0
4,0,3,35.0,0,0,8.0500,0,1,0,0,1,0,1,0,0,0,0,0,0


In [7]:
from sklearn.model_selection import train_test_split

X = df.drop(columns= "Survived")
y = df["Survived"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size= 0.2,
    random_state= 42
)

In [8]:
from sklearn.tree import DecisionTreeClassifier

model = DecisionTreeClassifier(max_depth = 3, random_state= 42)

model.fit(X_train, y_train)
y_pred = model.predict(X_test)


In [9]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall   :", recall_score(y_test, y_pred))
print("F1 Score :", f1_score(y_test, y_pred))

Accuracy : 0.7988826815642458
Precision: 0.796875
Recall   : 0.6891891891891891
F1 Score : 0.7391304347826086


In [10]:
print("Train Accuracy :", model.score(X_train, y_train))
print("Test Accuracy  :", model.score(X_test, y_test))

Train Accuracy : 0.8342696629213483
Test Accuracy  : 0.7988826815642458
